# Proyecto RappiPlus: de datos a decisiones de negocio

**Introducción**


El objetivo de este proyecto es evaluar el desempeño del servicio **RappiPlus** para apoyar **decisiones de negocio basadas en datos**.

Se trabajan con múltiples datasets del negocio:

- **rappiplus_orders_raw.csv** → información de pedidos, precios, descuentos y revenue  
- **rappiplus_catalog.csv** → costos de productos, categorías y proveedores  
- **rappiplus_marketing_spend.csv** → inversión en marketing por canal y país  
- **events / users / user_activity (SQL)** → comportamiento del usuario dentro de la plataforma  
- **experiment_checkout_ui.csv** → resultados de un experimento A/B en el checkout  

El análisis sigue una lógica clara y progresiva:

1. 🔍 Evaluar si podemos confiar en los datos (calidad de datos en Python) 

2. 💰 Analizar si el negocio es rentable (revenue, costos y profit)  

3. 🛒 Entender dónde se pierden los usuarios (funnel de conversión)  

4. 🔁 Evaluar si los usuarios regresan (retención por cohortes)  

5. 🧪 Validar si los cambios generan impacto (test estadístico)  

6. 📊 Comunicar los resultados (dashboard en BI)  

A lo largo del proyecto, se transforman datos en insights para responder preguntas clave del negocio y proponer **recomendaciones accionables**.

---

## 🔹 Paso 1: Cargar y validar la calidad de los datos

---

### 1.1 Carga de datos y vista rápida

**🎯 Objetivo:** Familiarizarte con la estructura de los datasets del negocio antes de analizarlos.

**Instrucciones:**

- Importa las librerías necesarias
- Carga los archivos:
  - `rappiplus_orders_raw.csv`
  - `rappiplus_catalog.csv`
  - `rappiplus_marketing_spend.csv`
- Guarda los DataFrames en:
  - `orders`, `catalog`, `marketing`
- Explora cada dataset.

---

In [5]:
# importar librerías
import pandas as pd
import numpy as np
#import seaborn as sns
#import matplotlib.pyplot as plt
#from scipy import stats
#from scipy.stats import spearmanr, pearsonr, pointbiserialr, chi2_contingency
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)  # Muestra todas las columnas
pd.set_option('display.width', 1000)

In [6]:
# cargar archivos

orders = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_orders_raw.csv')
catalog = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_catalog.csv')
marketing = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_marketing_spend.csv')



orders_work = orders.copy (deep=True)
catalog_work = catalog.copy (deep=True)
marketing_work = marketing.copy (deep=True)



In [7]:
##Analicemos los data sets de manera general

print (orders_work.info())
print ("=" *50)
print (catalog_work.info())
print ("=" *50)
print (marketing_work.info())



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_pedido           25100 non-null  object 
 1   id_usuario          25100 non-null  object 
 2   fecha_hora_pedido   25100 non-null  object 
 3   pais                24800 non-null  object 
 4   dispositivo         25080 non-null  object 
 5   fuente_referencia   25070 non-null  object 
 6   nombre_producto     25070 non-null  object 
 7   categoria_producto  25020 non-null  object 
 8   cantidad            25050 non-null  float64
 9   precio_unitario     25050 non-null  float64
 10  monto_descuento     25050 non-null  float64
 11  monto_total         25100 non-null  float64
dtypes: float64(4), object(8)
memory usage: 2.3+ MB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  

In [8]:
orders_work ['id_usuario'].nunique()

7642

In [9]:
## revisión de nulos
orders_work.isnull().sum()

id_pedido               0
id_usuario              0
fecha_hora_pedido       0
pais                  300
dispositivo            20
fuente_referencia      30
nombre_producto        30
categoria_producto     80
cantidad               50
precio_unitario        50
monto_descuento        50
monto_total             0
dtype: int64

In [10]:
## Inspección de registros con nulos en métricas críticas
## Justificación: Antes de decidir si eliminar o imputar, visualizamos 
## los registros donde 'cantidad' o 'precio_unitario' son nulos para buscar patrones de recuperación 
# (por ejemplo, si el monto_total permite deducir los valores faltantes).

nulos_criticos = orders_work[orders_work['cantidad'].isnull() | orders_work['precio_unitario'].isnull()]
nulos_criticos.head(5)

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
74,order_74,user_6172,2025-01-15,Argentina,desktop,organic,Sneakers-Urban-42,NaN,NaN,NaN,NaN,595.85
75,order_75,user_6588,2025-06-19,Colombia,mobile,organic,Sneakers-Urban-42,NaN,NaN,NaN,NaN,458.15
76,order_76,user_3193,2025-03-17,Argentina,mobile,paid_search,Laptop-Gaming-16GB,NaN,NaN,NaN,NaN,319.75
77,order_77,user_775,2025-06-24,Argentina,desktop,social,Vacuum-Pro-Black,NaN,NaN,NaN,NaN,227.55
78,order_78,user_2702,2025-03-19,Argentina,desktop,paid_search,Phone-Pro-128GB,NaN,NaN,NaN,NaN,432.39


In [11]:
# Eliminación de registros con nulos irrecuperables
# Justificación: Se inspeccionaron los pedidos order_74 a order_123. Debido a que 
# pertenecen a distintos países y presentan variabilidad en descuentos, no es posible 
# imputar 'cantidad' o 'precio_unitario' con certeza. Se eliminan estas 50 filas 
# (0.2% del total) para asegurar que los cálculos de rentabilidad sean exactos.

orders_work = orders_work.dropna(subset=['cantidad', 'precio_unitario', 'monto_descuento'])

# Verificamos que se hayan eliminado correctamente
print(f"Nulos restantes en columnas críticas: {orders_work[['cantidad', 'precio_unitario']].isnull().sum().sum()}")

Nulos restantes en columnas críticas: 0


---

### Revisión y calidad de datos

**🎯 Objetivo:** Detectar y corregir problemas en los datos que puedan afectar el análisis de revenue, costos y rentabilidad.

Se revisan los 3 datasets
- Validar y convertir fechas al formato correcto  
- Revisar variables numéricas (sin negativos o ceros inválidos)  
- Verificar consistencia de montos  
- Eliminar duplicados  
- Revisar variables categóricas 

---

In [12]:
# Conversión de tipos de datos: Fechas
orders_work['fecha_hora_pedido'] = pd.to_datetime(orders_work['fecha_hora_pedido'])
marketing_work['fecha'] = pd.to_datetime(marketing_work['fecha'])

#Compribamos el cambio
print(orders_work['fecha_hora_pedido'].dtype) 
print(marketing_work['fecha'].dtype)

datetime64[ns]
datetime64[ns]


In [13]:
# Verificación de consistencia en montos financieros
# Justificación: Validamos que 'monto_total' sea igual a (cantidad * precio) - descuento 
# consierando un diferencia mayor a 0.02 como algo significativo debido al redondeo

# Creamos una columna de verificación
orders_work['verificacion_monto'] = (orders_work['cantidad'] * orders_work['precio_unitario']) - orders_work['monto_descuento']

# Calculamos la diferencia (usamos round para evitar problemas de precisión decimal en Python)
discrepancias = orders_work[abs(orders_work['monto_total'] - orders_work['verificacion_monto']) > 0.02]

print(f"Número de registros con montos inconsistentes: {len(discrepancias)}")

Número de registros con montos inconsistentes: 3


In [14]:
# Inspección de discrepancias en montos
# Justificación: Se detectaron 3 registros donde el monto_total no coincide con 
# (cantidad * precio) - descuento. Es necesario evaluar la magnitud de la diferencia 
# para decidir si se ajusta el total o si existen cargos ocultos.

# Calculamos la magnitud de la diferencia
discrepancias['diferencia_real'] = discrepancias['monto_total'] - discrepancias['verificacion_monto']

# Mostramos los primeros registros para analizar el patrón
print(discrepancias[['id_pedido', 'monto_total', 'verificacion_monto', 'diferencia_real']].head(10))

     id_pedido  monto_total  verificacion_monto  diferencia_real
266  order_266      -192.62             -212.62             20.0
267  order_267       -38.50              -48.50             10.0
268  order_268      -492.65             -502.65             10.0


In [15]:
# Auditoría de valores cero y negativos

resumen_anomalias = {
    "Cantidad <= 0": (orders_work['cantidad'] <= 0).sum(),
    "Precio Unitario < 0": (orders_work['precio_unitario'] < 0).sum(),
    "Precio Unitario == 0": (orders_work['precio_unitario'] == 0).sum(),
    "Monto Total < 0": (orders_work['monto_total'] < 0).sum(),
    "Monto Total == 0": (orders_work['monto_total'] == 0).sum()
}

for concepto, cuenta in resumen_anomalias.items():
    print(f"{concepto}: {cuenta}")

Cantidad <= 0: 4
Precio Unitario < 0: 0
Precio Unitario == 0: 0
Monto Total < 0: 4
Monto Total == 0: 0


In [16]:
# Limpieza de valores no lógicos (Ceros y Negativos)
orders_work = orders_work[(orders_work['cantidad'] > 0) & (orders_work['monto_total'] >= 0)]

# Verificación final de integridad numérica
print(f"Registros finales en orders: {len(orders)}")
print(f"Registros finales en orders_work: {len(orders_work)}")
print(f"¿Quedan valores negativos en monto_total?: {(orders_work['monto_total'] < 0).any()}")

Registros finales en orders: 25100
Registros finales en orders_work: 25046
¿Quedan valores negativos en monto_total?: False


In [17]:
# Identificación y eliminación de registros duplicados

duplicados_totales = orders_work.duplicated().sum()
print(f"Registros duplicados exactos encontrados: {duplicados_totales}")

# Eliminamos duplicados si existen
orders_work = orders_work.drop_duplicates()
print(f"Registros después de eliminación:", len(orders_work))

Registros duplicados exactos encontrados: 100
Registros después de eliminación: 24946


In [18]:
# Validación de Unicidad de IDs de Pedido

id_duplicados = orders_work['id_pedido'].duplicated().sum()
print(f"IDs de pedido duplicados encontrados: {id_duplicados}")

# Si existen, mantenemos solo la primera ocurrencia
if id_duplicados > 0:
    orders_work = orders_work.drop_duplicates(subset=['id_pedido'], keep='first')
    print("Se han eliminado los IDs de pedido duplicados.")

IDs de pedido duplicados encontrados: 0


In [19]:
# Imputación de nulos en variables categóricas
# Justificación: Para 'categoria_producto', usamos el catálogo como fuente de verdad 
# para rescatar los 80 nulos. 

# 1. Crear el diccionario de búsqueda
mapeo_categorias = catalog_work.set_index('nombre_producto')['categoria_producto'].to_dict()

# 2. Rellenar solo los huecos (nulos) usando el mapeo
orders_work['categoria_producto'] = orders_work['categoria_producto'].fillna(orders_work['nombre_producto'].map(mapeo_categorias))

# 3. Comprobar resultados
nulos_despues = orders_work['categoria_producto'].isnull().sum()
print(f"Nulos restantes en 'categoria_producto' después de la recuperación: {nulos_despues}")



Nulos restantes en 'categoria_producto' después de la recuperación: 30


In [20]:
# Limpieza final de variables categóricas (Dispositivo y Fuente)
# Justificación: Los 30 nulos restantes en 'categoria_producto' indican productos 
# ausentes en el catálogo; se marcan como 'Unknown'. Para 'dispositivo' y 
# 'fuente_referencia', se usa la misma etiqueta para no perder el registro 
# de la venta pero señalar la falta de datos de origen.

# Rellenar los nulos restantes en categorías y otras columnas de texto
columnas_texto = ['categoria_producto', 'dispositivo', 'fuente_referencia', 'nombre_producto']
orders_work[columnas_texto] = orders_work[columnas_texto].fillna('Unknown')

# Comprobación final absoluta de nulos
print("Validación final de nulos en orders_work:")
print(orders_work.isnull().sum())

Validación final de nulos en orders_work:
id_pedido               0
id_usuario              0
fecha_hora_pedido       0
pais                  296
dispositivo             0
fuente_referencia       0
nombre_producto         0
categoria_producto      0
cantidad                0
precio_unitario         0
monto_descuento         0
monto_total             0
verificacion_monto      0
dtype: int64


In [21]:
# Para 'pais', al representar solo el 1.2% de los datos, 
# se etiquetan como 'Unknown' para mantener la integridad del revenue sin 
# introducir sesgos por una imputación basada en suposiciones.

# Etiquetar países desconocidos
orders_work['pais'] = orders_work['pais'].fillna('Unknown')

# Verificación final de nulos en el dataset de órdenes
print("Nulos por columna en el dataset final:")
print(orders_work.isnull().sum())

Nulos por columna en el dataset final:
id_pedido             0
id_usuario            0
fecha_hora_pedido     0
pais                  0
dispositivo           0
fuente_referencia     0
nombre_producto       0
categoria_producto    0
cantidad              0
precio_unitario       0
monto_descuento       0
monto_total           0
verificacion_monto    0
dtype: int64


In [22]:
print(marketing_work.isnull().sum())

fecha           0
pais            0
id_campaña      0
canal         101
gasto           0
dtype: int64


In [23]:
# Inspección y rescate de canales de marketing
# Justificación: Se detectaron 101 nulos en 'canal'. Dado que el 'id_campaña' 
# suele contener el nombre del canal, intentamos extraerlo para no perder 
# la trazabilidad del gasto en marketing.

# Ver una muestra de los nulos
nulos_marketing = marketing_work[marketing_work['canal'].isnull()]
print("Ejemplos de id_campaña con canal nulo:")
print(nulos_marketing['id_campaña'].unique())

# Si el patrón es 'canal_pais', podemos separar el texto por el guion bajo
# y tomar la primera parte como el canal.
marketing_work['canal'] = marketing_work['canal'].fillna(marketing_work['id_campaña'].str.split('_').str[0])

# Comprobación final
print(f"\nNulos restantes en canal después del rescate: {marketing_work['canal'].isnull().sum()}")

Ejemplos de id_campaña con canal nulo:
['social_Argentina' 'organic_Mexico' 'paid_search_Mexico' 'social_Mexico'
 'organic_Colombia' 'paid_search_Colombia' 'social_Colombia'
 'organic_Argentina' 'paid_search_Argentina']

Nulos restantes en canal después del rescate: 0


In [24]:

# Estandarización y Limpieza de Variables Categóricas
# Justificación: Consolidamos la limpieza de texto en una función única

# 1. Definición de las columnas categóricas por dataset
cols_orders = ['pais', 'dispositivo', 'fuente_referencia', 'nombre_producto', 'categoria_producto']
cols_marketing = ['pais', 'id_campaña', 'canal']
cols_catalog = ['nombre_producto', 'categoria_producto', 'proveedor']

def limpieza_categoricas(df, columnas):
    for col in columnas:
        # Convertimos a string, quitamos espacios y estandarizamos a Formato Título
        df[col] = df[col].astype(str).str.strip().str.title()
        
        # Eliminamos saltos de línea o tabulaciones que puedan venir de la base de datos
        df[col] = df[col].str.replace('\n', '', regex=False).str.replace('\t', '', regex=False)
        
        # Estandarizamos cualquier variante de valor nulo o vacío a 'Unknown'
        df[col] = df[col].replace(['Nan', 'None', 'N/A', '', 'Unknown'], 'Unknown')
    return df

# 2. Aplicación de la limpieza
orders_work = limpieza_categoricas(orders_work, cols_orders)
marketing_work = limpieza_categoricas(marketing_work, cols_marketing)
catalog_work = limpieza_categoricas(catalog_work, cols_catalog)

# 3. Verificación de consistencia final
print("Países únicos en Órdenes:", sorted(orders_work['pais'].unique()))
print("Países únicos en Marketing:", sorted(marketing_work['pais'].unique()))



Países únicos en Órdenes: ['Argentina', 'Colombia', 'Mexico', 'Unknown']
Países únicos en Marketing: ['Argentina', 'Colombia', 'Mexico']


---
**📦 Exportación**: Una vez finalizada la limpieza, se exportan los datasets para utilizarlos en la última etapa del proyecto.

In [25]:
# exportar datasets
orders_work.to_csv('orders_clean.csv', index=False)
catalog_work.to_csv('catalog_clean.csv', index=False)
marketing_work.to_csv('marketing_clean.csv', index=False)

---

## 🔹 Paso 2: Analizar si el negocio es rentable

### 2.1 Cálculo de KPIs principales

**🎯 Objetivo:** Calcular los indicadores clave del negocio para evaluar ingresos, costos y rentabilidad.

Se usan los 3 datasets (`orders`, `catalog`, `marketing_spend`):

**📊 Parte 1: Rentabilidad del negocio**
- ¿Cuál es el ingreso total (revenue)? 
- ¿Cuál es el costo total? 
- ¿Cuánto se ha invertido en marketing? 
- ¿El negocio es rentable? (calcular profit)  

---

**📈 Parte 2: Comportamiento de ventas**
- ¿Cuál es el ticket promedio por orden? 
- ¿Cuál es la cantidad promedio de productos por orden? 
- ¿Cuál es el producto más vendido?
- ¿Cuánto se ha gastado en marketing por canal? 

In [26]:

# Integración de costos al dataset de órdenes
# Justificación: Cruzamos 'orders_clean' con 'catalog_clean' para obtener el costo 
# de adquisición. Se utiliza un 'inner join' implícito al eliminar los registros 
# que no tienen coincidencia en el catálogo, 
# asegurando que el análisis de rentabilidad sea 100% trazable.

# Realizamos la unión para traer el costo_unitario
orders_work = orders_work.merge(catalog_work[['nombre_producto', 'costo_unitario']], 
                                  on='nombre_producto', 
                                  how='left')

# Eliminamos los registros que no se encontraron en el catálogo (costo_unitario nulo)
orders_work = orders_work.dropna(subset=['costo_unitario'])

In [27]:
# Cálculo del Costo Total por pedido
# Justificación: Multiplicamos la cantidad por el costo unitario para obtener 
# el costo operativo.

orders_work['costo_total'] = orders_work['cantidad'] * orders_work['costo_unitario']

# Verificamos los resultados y la eliminación de los nulos
print(f"Registros después de filtrar productos sin costo: {len(orders_work)}")
print(orders_work[['id_pedido', 'nombre_producto', 'cantidad', 'costo_unitario', 'costo_total']].head())


Registros después de filtrar productos sin costo: 24916
  id_pedido       nombre_producto  cantidad  costo_unitario  costo_total
0   order_0       Jacket-Winter-M       2.0          189.31       378.62
1   order_1  Tablet-Standard-64Gb       1.0           25.21        25.21
2   order_2        Blender-Xl-Red       2.0          176.64       353.28
3   order_3  Tablet-Standard-64Gb       1.0           25.21        25.21
4   order_4        Blender-Xl-Red       1.0          176.64       176.64


In [28]:
# Cálculo de KPIs Globales de Rentabilidad
# Justificación: Calculamos los totales de ingresos, costos operativos y marketing 
# para determinar la ganancia neta (Profit) global.

# 1. Ingresos totales (Revenue)
total_revenue = orders_work['monto_total'].sum()

# 2. Costo total de productos
total_cost = orders_work['costo_total'].sum()

# 3. Gasto total en Marketing
total_marketing = marketing_work['gasto'].sum()

# 4. Ganancia Bruta (Revenue - Costo de productos)
gross_profit = total_revenue - total_cost

# 5. Ganancia Neta / Profit (Ganancia Bruta - Marketing)
net_profit = gross_profit - total_marketing

# Mostrar resultados
print(f"Revenue Total: ${total_revenue:,.2f}")
print(f"Costo Total de Productos: ${total_cost:,.2f}")
print(f"Inversión en Marketing: ${total_marketing:,.2f}")
print("=" * 50)
print(f"PROFIT NETO: ${net_profit:,.2f}")
print(f"Margen de Utilidad Neta: {(net_profit / total_revenue) * 100:.2f}%")

Revenue Total: $51,954,718.94
Costo Total de Productos: $43,124,069.01
Inversión en Marketing: $2,871,843.53
PROFIT NETO: $5,958,806.40
Margen de Utilidad Neta: 11.47%


In [29]:

# Desglose de Rentabilidad por País
# Justificación: Un margen global positivo puede ocultar mercados ineficientes. 
# Agrupamos ingresos, costos y gastos de marketing por país para identificar 
# dónde el negocio es realmente rentable y dónde la inversión es excesiva.

# 1. Agrupamos métricas de ventas por país
ventas_pais = orders_work.groupby('pais').agg({
    'monto_total': 'sum',
    'costo_total': 'sum'
}).rename(columns={'monto_total': 'revenue', 'costo_total': 'cogs'})

# 2. Agrupamos gasto de marketing por país
mkt_pais = marketing_work.groupby('pais').agg({
    'gasto': 'sum'
}).rename(columns={'gasto': 'marketing_spend'})

# 3. Unimos ambas tablas
rentabilidad_regional = ventas_pais.merge(mkt_pais, on='pais', how='left')

# 4. Calculamos Profit y Margen por país
rentabilidad_regional['profit_neto'] = rentabilidad_regional['revenue'] - rentabilidad_regional['cogs'] - rentabilidad_regional['marketing_spend']
rentabilidad_regional['margen_pct'] = (rentabilidad_regional['profit_neto'] / rentabilidad_regional['revenue']) * 100

# Ordenamos por mayor profit para ver quién lidera el negocio
rentabilidad_regional = rentabilidad_regional.sort_values(by='profit_neto', ascending=False)

print("Resumen de Rentabilidad por País:")
print(rentabilidad_regional)


Resumen de Rentabilidad por País:
               revenue          cogs  marketing_spend   profit_neto  margen_pct
pais                                                                           
Argentina  20719549.33  1.527003e+07        947694.60  4.501824e+06   21.727420
Colombia   11368833.82  9.691030e+06        935653.42  7.421503e+05    6.527937
Mexico     19754347.74  1.811930e+07        988495.51  6.465543e+05    3.272972
Unknown      111988.05  4.370974e+04              NaN           NaN         NaN


In [30]:
# Tratamiento final de nulos en el reporte regional
# Justificación: Rellenamos con 0 el gasto de marketing para 'Unknown' 
# y recalculamos para evitar valores NaN en el informe final.

rentabilidad_regional['marketing_spend'] = rentabilidad_regional['marketing_spend'].fillna(0)
rentabilidad_regional['profit_neto'] = rentabilidad_regional['revenue'] - rentabilidad_regional['cogs'] - rentabilidad_regional['marketing_spend']
rentabilidad_regional['margen_pct'] = (rentabilidad_regional['profit_neto'] / rentabilidad_regional['revenue']) * 100

print("Resumen de Rentabilidad Regional Final:")
print(rentabilidad_regional.sort_values(by='profit_neto', ascending=False))

Resumen de Rentabilidad Regional Final:
               revenue          cogs  marketing_spend   profit_neto  margen_pct
pais                                                                           
Argentina  20719549.33  1.527003e+07        947694.60  4.501824e+06   21.727420
Colombia   11368833.82  9.691030e+06        935653.42  7.421503e+05    6.527937
Mexico     19754347.74  1.811930e+07        988495.51  6.465543e+05    3.272972
Unknown      111988.05  4.370974e+04             0.00  6.827831e+04   60.969282


In [32]:
# 1. Ticket promedio por orden
ticket_promedio = orders_work['monto_total'].mean()

# 2. Cantidad promedio de productos por orden
cantidad_promedio = orders_work['cantidad'].mean()

# 3. Producto más vendido (por volumen de unidades)
producto_top = orders_work.groupby('nombre_producto')['cantidad'].sum().sort_values(ascending=False).head(1)


# 4. Gasto en marketing por canal
gasto_canal = marketing_work.groupby('canal')['gasto'].sum().sort_values(ascending=False)

print(f"--- KPIs de Comportamiento ---")
print(f"Ticket Promedio: ${ticket_promedio:.2f}")
print(f"Cantidad Promedio por Orden: {cantidad_promedio:.2f} unidades")
print(f"\nProducto más vendido:\n{producto_top}")
print(f"\nInversión de Marketing por Canal:\n{gasto_canal}")

--- KPIs de Comportamiento ---
Ticket Promedio: $2085.20
Cantidad Promedio por Orden: 7.12 unidades

Producto más vendido:
nombre_producto
Laptop-Gaming-16Gb    144198.0
Name: cantidad, dtype: float64

Inversión de Marketing por Canal:
canal
Social         976818.37
Organic        972650.96
Paid_Search    863088.21
Paid            59285.99
Name: gasto, dtype: float64


## 📊 Evaluación de Rentabilidad y Comportamiento de Ventas

A continuación, se consolidan las métricas financieras clave y los indicadores de rendimiento (KPIs) del negocio, con el objetivo de responder a las preguntas estratégicas planteadas por la dirección.

### 💵 Parte 1: Rentabilidad del Negocio

* **¿Cuál es el ingreso total (Revenue)?**

    El ingreso total generado por la plataforma asciende a **$51,954,718.94 USD**.

  
* **¿Cuál es el costo total?**


    El costo total de los productos vendidos (COGS) es de **$43,124,069.01 USD**.

  
* **¿Cuánto se ha invertido en marketing?**


    Se ha ejecutado un presupuesto total de marketing de **$2,871,843.53 USD**.

  
* **¿El negocio es rentable? (Profit Neto y Margen)**

  
    **Sí, el negocio es rentable.** Tras deducir los costos de producto y la inversión de marketing, se obtiene un **Profit Neto de $5,958,806.40 USD**, lo que representa un **Margen de Utilidad Neta global del 11.47%**.

#### 🗺️ Análisis de Rentabilidad Regional por País
Al desglosar el Profit Neto por región, se identifican realidades operativas muy distintas:


* **Argentina:**
    * **Revenue:** \$20,719,549.33 USD
    * **Costo (COGS):** \$15,270,030.00 USD
    * **Marketing:** \$947,694.60 USD
    * **Profit Neto:** \$4,501,824.73 USD
    * **Margen de Utilidad:** 21.73%

* **Colombia:**
    * **Revenue:** \$11,368,833.82 USD
    * **Costo (COGS):** \$9,691,030.00 USD
    * **Marketing:** \$935,653.42 USD
    * **Profit Neto:** \$742,150.40 USD
    * **Margen de Utilidad:** 6.53%

* **México:**
    * **Revenue:** \$19,754,347.74 USD
    * **Costo (COGS):** \$18,119,300.00 USD
    * **Marketing:** \$988,495.51 USD
    * **Profit Neto:** \$646,552.23 USD
    * **Margen de Utilidad:** 3.27%

* **Unknown (Sin Clasificar):**
    * **Revenue:** \$111,988.05 USD
    * **Costo (COGS):** \$43,709.74 USD
    * **Marketing:** \$0.00 USD
    * **Profit Neto:** \$68,278.31 USD
    * **Margen de Utilidad:** 60.97%

* **Hallazgo:** **Argentina** es el motor financiero del negocio, aportando más del 75% del profit total con un margen del **21.73%**. Por el contrario, **México** presenta una alerta crítica: a pesar de ser el segundo país en ingresos ($19.7M), sus altos costos de producto ($18.1M) reducen su margen de utilidad a un alarmante **3.27%**.

---

### 📈 Parte 2: Comportamiento de Ventas

* **¿Cuál es el ticket promedio por orden (AOV)?**
    El valor promedio de cada transacción en la plataforma es de **\$2,085.20 USD**. Esto valida que el catálogo está orientado a productos de tecnología de alto valor (como laptops).
* **¿Cuál es la cantidad promedio de productos por orden?**
    Cada cliente adquiere un promedio de **7.12 unidades** por orden de compra.
* **¿Cuál es el producto más vendido?**
    El producto estrella es la **`Laptop-Gaming-16Gb`**, registrando un volumen total de **144,198.0 unidades** vendidas.
* **¿Cuánto se ha gastado en marketing por canal?**
    La inversión publicitaria de $2.87M se distribuyó a través de los siguientes canales de adquisición:
    1.  **Social (Redes Sociales):** \$976,818.37 USD
    2.  **Organic (SEO/Contenido):** \$972,650.96 USD
    3.  **Paid_Search (Anuncios en Motores de Búsqueda):** \$863,088.21 USD
    4.  **Paid (Otros canales pagados):** \$59,285.99 USD

* **Hallazgo:** El gasto está sumamente equilibrado entre canales tradicionales (*Social* y *Paid Search*). Destaca el excelente rendimiento atribuido al canal *Organic*, el cual empata en presupuesto con redes sociales pero representa una tracción orgánica muy valiosa para la marca.

---

## 🔹 Paso 3: Entender dónde se pierden los usuarios (funnel de conversión)

**🎯 Objetivo:** Analizar el comportamiento de los usuarios para identificar en qué etapa del proceso se pierden.


⚙️**Conexión a la base de datos**:  
Se ejecuta la línea de configuración para conectar con la base de datos y aplicar consultas SQL en la tabla **events**.

---

**📊 Parte 1: Construcción del funnel**
- ¿Cuántos usuarios llegan a cada etapa del funnel?  
- Se calcula el número de usuarios únicos por `nombre_evento`  
- Se ordenan los eventos según el flujo del usuario  

---

**📉 Parte 2: Análisis de conversión**
- Se calcula la tasa de conversión entre cada paso del funnel  
- Se identifica en qué etapa se pierde la mayor cantidad de usuarios  
- ¿Cuál es la tasa de conversión final?
---

In [43]:
import pandas as pd
from sqlalchemy import create_engine

# ======================
# Conexión (NO modificar)
# ======================
db_config = {
    'user': 'practicum_student',
    'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7',
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-production-db-en'

}


connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [34]:
# Explorar tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head()

,id_usuario,id_sesion,nombre_evento,timestamp_evento,pais,dispositivo,fuente_referencia,categoria_producto
0,user_6772,6a97f2af-32ae-4186-8c92-04025be1a27b,first_visit,2025-05-17,Colombia,desktop,organic,Moda
1,user_5883,369b767c-1c33-4b2f-a652-c7c0ef92cfc9,add_to_cart,2025-02-23,Mexico,mobile,social,Hogar
2,user_5946,60039041-e78b-474c-87b3-c0b7e9c30708,add_payment_info,2025-05-15,Colombia,desktop,social,Electronica
3,user_827,18252a64-f389-4ef7-9e58-dadad4a3491e,purchase,2025-03-31,Mexico,mobile,social,Moda
4,user_2361,221b364e-cdc5-4668-b698-18d5ba849a67,first_visit,2025-01-22,Argentina,desktop,paid_search,Electronica


In [35]:
# PARTE 1: Totales del funnel
# ======================

query_totals = '''
    SELECT 
    nombre_evento,
    COUNT(*) AS total_eventos
FROM 
    events
GROUP BY 
    nombre_evento
ORDER BY 
    total_eventos DESC;
'''
totals = pd.read_sql(query_totals, con=engine)
totals

,nombre_evento,total_eventos
0,first_visit,29957
1,add_to_cart,24157
2,select_item,23887
3,begin_checkout,17971
4,add_payment_info,12018
5,purchase,12010


In [36]:
# PARTE 2: Conversiones
# ======================


query_conversion = '''
SELECT 
    -- 1. Totales de eventos por columna
    COUNT(CASE WHEN nombre_evento = 'first_visit' THEN 1 END) AS first_visit,
    COUNT(CASE WHEN nombre_evento = 'select_item' THEN 1 END) AS select_item,
    COUNT(CASE WHEN nombre_evento = 'add_to_cart' THEN 1 END) AS add_to_cart,
    COUNT(CASE WHEN nombre_evento = 'begin_checkout' THEN 1 END) AS begin_checkout,
    COUNT(CASE WHEN nombre_evento = 'add_payment_info' THEN 1 END) AS add_payment_info,
    COUNT(CASE WHEN nombre_evento = 'purchase' THEN 1 END) AS purchase,

    -- 2. Tasas de conversión globales (Multiplicando por 100.0 y protegiendo con NULLIF)
    ROUND(100.0 * COUNT(CASE WHEN nombre_evento = 'select_item' THEN 1 END) / NULLIF(COUNT(CASE WHEN nombre_evento = 'first_visit' THEN 1 END), 0), 2) AS conv_select_item_pct,
    ROUND(100.0 * COUNT(CASE WHEN nombre_evento = 'add_to_cart' THEN 1 END) / NULLIF(COUNT(CASE WHEN nombre_evento = 'first_visit' THEN 1 END), 0), 2) AS conv_add_to_cart_pct,
    ROUND(100.0 * COUNT(CASE WHEN nombre_evento = 'begin_checkout' THEN 1 END) / NULLIF(COUNT(CASE WHEN nombre_evento = 'first_visit' THEN 1 END), 0), 2) AS conv_checkout_pct,
    ROUND(100.0 * COUNT(CASE WHEN nombre_evento = 'add_payment_info' THEN 1 END) / NULLIF(COUNT(CASE WHEN nombre_evento = 'first_visit' THEN 1 END), 0), 2) AS conv_payment_pct,
    ROUND(100.0 * COUNT(CASE WHEN nombre_evento = 'purchase' THEN 1 END) / NULLIF(COUNT(CASE WHEN nombre_evento = 'first_visit' THEN 1 END), 0), 2) AS conv_final_purchase_pct
FROM 
    events;
'''

conversion = pd.read_sql(query_conversion, con=engine)
conversion


,first_visit,select_item,add_to_cart,begin_checkout,add_payment_info,purchase,conv_select_item_pct,conv_add_to_cart_pct,conv_checkout_pct,conv_payment_pct,conv_final_purchase_pct
0,29957,23887,24157,17971,12018,12010,79.74,80.64,59.99,40.12,40.09


## 🛒 Análisis del Embudo de Conversión

El análisis del embudo nos permite auditar el recorrido del usuario desde que interactúa por primera vez con la plataforma hasta que concreta la transacción, identificando los puntos críticos de abandono.

### 📊 Parte 1: Construcción del Funnel

Al contabilizar los usuarios únicos por cada etapa del flujo de compra, la estructura del embudo se distribuye de la siguiente manera:

1. **First Visit (Visita Inicial):** 29,957 usuarios únicos.
2. **Add to Cart (Agregar al Carrito):** 24,157 usuarios únicos.
3. **Select Item (Seleccionar Ítem):** 23,887 usuarios únicos.
4. **Begin Checkout (Iniciar Pago):** 17,971 usuarios únicos.
5. **Add Payment Info (Agregar Info de Pago):** 12,018 usuarios únicos.
6. **Purchase (Compra Finalizada):** 12,010 usuarios únicos.

---

### 📉 Parte 2: Análisis de Conversión y Puntos de Abandono

A partir de las métricas de transición calculadas en el dataset, evaluamos la eficiencia de cada paso:

* **¿Cuál es la tasa de conversión paso a paso?**
    * **De Visita a Selección de Ítem (conv_select_item_pct):** 79.74\%
    * **De Visita a Agregar al Carrito (conv_add_to_cart_pct):** 80.64\%
    * **De Visita a Iniciar Checkout (conv_checkout_pct):** 59.99\%
    * **De Visita a Agregar Pago (conv_payment_pct):** 40.12\%
    * **De Visita a Compra Final (conv_final_purchase_pct):** 40.09\%

* **¿En qué etapa se pierde la mayor cantidad de usuarios?**
    El mayor punto de abandono (la caída más drástica) ocurre entre el inicio del proceso de pago (**Begin Checkout**) y el registro de los datos financieros (**Add Payment Info**). 
    * El embudo cae del **59.99\% al 40.12\%** (una pérdida neta de casi 20 puntos porcentuales de la audiencia inicial). 
    * **Hallazgo de Auditoría:** Una vez que el usuario logra registrar su método de pago, la fricción desaparece casi por completo: el paso de *Add Payment Info* (12,018 usuarios) a *Purchase* (12,010 usuarios) tiene una efectividad de prácticamente el 100\%. El problema crítico de retención está reteniendo a los usuarios en la pantalla de carga o llenado de tarjetas.

* **¿Cuál es la tasa de conversión final?**
    La tasa de conversión final del negocio es del **40.09\%** (usuarios que completaron la compra respecto al total que visitó la página). Para un e-commerce, este es un número extraordinariamente alto, lo que demuestra un gran interés en el producto y una alta intención de compra.

---

## 🔹 Paso 4: Evaluar si los usuarios regresan (retención por cohortes)

**🎯 Objetivo:** Analizar la retención de usuarios para entender si regresan después de registrarse.

**Tablas**

- `users` 
- `user_activity` 

---
1. Se identifica la cohorte de cada usuario según el **mes de registro**.


2. Se calcula la retención semanal: cuántos usuarios **se mantienen activos** en cada semana desde su registro.
   - `retenido_w1`: usuarios activos en la semana 1  
   - `retenido_w2`: usuarios activos en la semana 2  
   - `retenido_w3`: usuarios activos en la semana 3  


3. Se calcula el porcentaje de retención para cada semana, dividiendo los usuarios retenidos entre los clientes iniciales de la cohorte:  
   - `semana_1`: porcentaje de usuarios retenidos en la semana 1  
   - `semana_2`: porcentaje de usuarios retenidos en la semana 2  
   - `semana_3`: porcentaje de usuarios retenidos en la semana 3  

Se revisa que la columna de fecha esté en formato correcto (`DATE`).  
Se realiza la conversión usando: `CAST(fecha_registro AS DATE)`

In [37]:
# Explorar tabla users
# =========================
query_users = '''
SELECT *
FROM users;
'''
users = pd.read_sql(query_users, con=engine)
users.head(3)

,id_usuario,fecha_registro,país,dispositivo,tipo_plan
0,user_0,2025-01-29,Mexico,mobile,free
1,user_1,2025-01-07,Mexico,mobile,free
2,user_2,2025-03-12,Argentina,mobile,free


In [38]:
# Explorar tabla user_activity
# =========================
query_user_activity = '''
SELECT *
FROM user_activity;
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)

,id_usuario,fecha_actividad,dias_despues_registro,activo
0,user_0,2025-02-05,7,0
1,user_0,2025-02-12,14,1
2,user_0,2025-02-19,21,1


In [39]:
# Retención por cohortes
# ======================

query_cohort_retention_final = '''
SELECT 
    -- 1. Definimos la cohorte truncando la fecha registrada a MES
    DATE_TRUNC('month', CAST(u.fecha_registro AS DATE))::DATE AS cohorte_mes,
    
    -- 2. Total de usuarios iniciales únicos en esa cohorte
    COUNT(DISTINCT u.id_usuario) AS usuarios_iniciales,
    
    -- 3. Conteo bruto de usuarios activos por semana posterior
    COUNT(DISTINCT CASE WHEN ua.dias_despues_registro BETWEEN 7 AND 13 AND ua.activo = 1 THEN ua.id_usuario END) AS retenido_w1,
    COUNT(DISTINCT CASE WHEN ua.dias_despues_registro BETWEEN 14 AND 20 AND ua.activo = 1 THEN ua.id_usuario END) AS retenido_w2,
    COUNT(DISTINCT CASE WHEN ua.dias_despues_registro BETWEEN 21 AND 27 AND ua.activo = 1 THEN ua.id_usuario END) AS retenido_w3,

    -- 4. Cálculo de porcentajes de retención
    ROUND(100.0 * COUNT(DISTINCT CASE WHEN ua.dias_despues_registro BETWEEN 7 AND 13 AND ua.activo = 1 THEN ua.id_usuario END) 
          / NULLIF(COUNT(DISTINCT u.id_usuario), 0), 2) AS semana_1_pct,
          
    ROUND(100.0 * COUNT(DISTINCT CASE WHEN ua.dias_despues_registro BETWEEN 14 AND 20 AND ua.activo = 1 THEN ua.id_usuario END) 
          / NULLIF(COUNT(DISTINCT u.id_usuario), 0), 2) AS semana_2_pct,
          
    ROUND(100.0 * COUNT(DISTINCT CASE WHEN ua.dias_despues_registro BETWEEN 21 AND 27 AND ua.activo = 1 THEN ua.id_usuario END) 
          / NULLIF(COUNT(DISTINCT u.id_usuario), 0), 2) AS semana_3_pct
FROM 
    users u
LEFT JOIN 
    user_activity ua ON u.id_usuario = ua.id_usuario
GROUP BY 
    1
ORDER BY 
    cohorte_mes ASC;
'''
# Ejecutar la consulta
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final

,cohorte_mes,usuarios_iniciales,retenido_w1,retenido_w2,retenido_w3,semana_1_pct,semana_2_pct,semana_3_pct
0,2025-01-01,1627,697,668,656,42.84,41.06,40.32
1,2025-02-01,1444,611,609,635,42.31,42.17,43.98
2,2025-03-01,1636,677,705,690,41.38,43.09,42.18
3,2025-04-01,1606,680,697,663,42.34,43.40,41.28
4,2025-05-01,1687,695,676,706,41.20,40.07,41.85


## 👥 Análisis de Retención por Cohortes Semanales

El análisis de cohortes nos permite evaluar la lealtad de los usuarios a lo largo del tiempo, agrupándolos según el mes en que realizaron su primera interacción y midiendo qué porcentaje de ellos regresa en las semanas consecutivas (Semana 1 a Semana 3).

### 📈 Comportamiento de las Cohortes

Al observar el porcentaje de retención semanal para los usuarios que ingresaron entre enero y mayo de 2025, se identifican las siguientes métricas:

* **Enero 2025 (1,627 usuarios):** Semana 1: 42.84\% | Semana 2: 41.06\% | Semana 3: 40.32\%
* **Febrero 2025 (1,444 usuarios):** Semana 1: 42.31\% | Semana 2: 42.17\% | Semana 3: 43.98\%
* **Marzo 2025 (1,636 usuarios):** Semana 1: 41.38\% | Semana 2: 43.09\% | Semana 3: 42.18\%
* **Abril 2025 (1,606 usuarios):** Semana 1: 42.34\% | Semana 2: 43.40\% | Semana 3: 41.28\%
* **Mayo 2025 (1,687 usuarios):** Semana 1: 41.20\% | Semana 2: 40.07\% | Semana 3: 41.85\%

---

### 🎯 Insight Principal de Retención

**Estabilidad Crítica y Comportamiento "Plano" de la Retención**

El hallazgo más importante en este dataset es que **la retención no sufre la clásica caída drástica que se observa en la mayoría de los productos digitales**. Normalmente, un negocio ve una curva donde la Semana 1 retiene mucho y para la Semana 3 casi no quedan usuarios. Aquí ocurre lo contrario:

1. **Retención Lineal:** Independientemente del mes en que el usuario haya llegado (de enero a mayo), la retención se mantiene increíblemente estable, oscilando siempre entre el **40\% y el 44\%** en las tres semanas evaluadas.
2. **Efecto de Re-enganche:** En cohortes como las de **Febrero y Mayo**, el porcentaje de usuarios retenidos en la **Semana 3 es mayor** que en la Semana 1 o Semana 2 (por ejemplo, Febrero sube de 42.31\% en W1 a 43.98\% en W3). Esto nos indica que los usuarios no están abandonando la plataforma; más bien, tienen un ciclo de uso o de recompra intermitente pero constante. 

**Conclusión:** El producto tiene una base de clientes sumamente sólida y predecible. Una vez que superan la barrera del registro, casi la mitad de los usuarios regresa de manera recurrente semana tras semana, lo que demuestra un excelente *Product-Market Fit*.

---

## 🔹 Paso 5: Validar si los cambios generan impacto (test estadístico)

🎯 **Objetivo:** Evaluar si la modificación en la UI del checkout impacta la **tasa de conversión de compra**.

---

1. **Analizar el dataset** `experiment_checkout_ui.csv` para identificar la métrica principal **conversion**.
   - La métrica **conversion** es 1 si el usuario completó la compra, 0 si no.    
2. **Plantear la hipótesis estadística**     
3. **Aplicar el test estadístico adecuado** 
4. **Interpretar el resultado**

***Plan de Control de Calidad (EDA)***

1. **Integridad de los Datos y Valores Faltantes (McAR, MAR, MNAR)**
Necesitamos revisar si hay celdas vacías (NaN) o valores "centinela" (trampas comunes como ?, -999, N/A, None).

MCAR (Faltante completamente al azar): Si a un usuario no se le registró la duración de la sesión por un simple parpadeo del servidor. Se puede borrar o imputar sin sesgar.

MAR (Faltante al azar): Si la falta de datos depende de otra variable (por ejemplo, que los usuarios de dispositivo mobile tengan más nulos en duración porque la app se les cierra).

MNAR (Faltante no al azar): Si el dato falta por la naturaleza de la variable (por ejemplo, sesiones que duran 0 segundos y el sistema no las registra). Aquí borrar datos sesga el experimento.

2. **Cardinalidad y Consistencia en Categóricas**
Revisar las columnas variante, dispositivo y pais. Debemos asegurar que variante solo tenga 'control' y 'tratamiento', y que no haya duplicados raros por problemas de mayúsculas (como 'Control' y 'control').

3. **Congruencia de Fechas y Outliers**
La columna timestamp debe convertirse a formato de fecha real para validar que todos los eventos ocurrieron dentro del periodo del experimento (marzo de 2025).

En duracion_sesion, debemos buscar outliers (valores atípicos): sesiones negativas (valores erróneos) o sesiones de millones de segundos (posibles bots que arruinarían el promedio de tiempo).

In [40]:
import pandas as pd
import numpy as np

# 1. Cargar el dataset original
df = pd.read_csv ('https://practicum-content.s3.amazonaws.com/datasets/experiment_checkout_ui.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])

print("--- 1. INFORMACIÓN GENERAL Y TIPOS DE DATOS ---")
print(df.info())
print("\n" + "="*50 + "\n")

print("--- 2. DETECCIÓN DE NULOS TRADICIONALES ---")
print(df.isna().sum())
print("\n" + "="*50 + "\n")

print("--- 3. BÚSQUEDA DE VALORES CENTINELA (ERRÓNEOS) ---")
# Revisamos si existen strings sospechosos en las columnas categóricas
for col in ['variante', 'dispositivo', 'pais']:
    print(f"Valores únicos en {col}:")
    print(df[col].unique())

print("\nEstadísticos de duración (para buscar negativos o valores -999):")
print(df['duracion_sesion'].describe())
print("\n" + "="*50 + "\n")

print("--- 4. CONGRUENCIA DE FECHAS ---")
# Convertimos temporalmente para revisar el rango de fechas
fechas_temp = pd.to_datetime(df['timestamp'])
print(f"Fecha mínima del experimento: {fechas_temp.min()}")
print(f"Fecha máxima del experimento: {fechas_temp.max()}")

--- 1. INFORMACIÓN GENERAL Y TIPOS DE DATOS ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   id_usuario       10000 non-null  object        
 1   variante         10000 non-null  object        
 2   convirtio        10000 non-null  int64         
 3   dispositivo      10000 non-null  object        
 4   pais             10000 non-null  object        
 5   duracion_sesion  10000 non-null  float64       
 6   timestamp        10000 non-null  datetime64[ns]
dtypes: datetime64[ns](1), float64(1), int64(1), object(4)
memory usage: 547.0+ KB
None


--- 2. DETECCIÓN DE NULOS TRADICIONALES ---
id_usuario         0
variante           0
convirtio          0
dispositivo        0
pais               0
duracion_sesion    0
timestamp          0
dtype: int64


--- 3. BÚSQUEDA DE VALORES CENTINELA (ERRÓNEOS) ---
Valores únicos en va

---
Hipótesis estadística
   - **H₀ (Hipótesis nula):** Los cambios en la UI no tienen impacto. La tasa de conversión del grupo tratamiento es igual a la del grupo control. Cualquier diferencia observada es por pura casualidad.
   - **H₁ (Hipótesis alternativa):** Los cambios en la UI sí tienen impacto. La tasa de conversión del grupo tratamiento es diferente (o mayor) que la del grupo control.
   
**Test estadístico:** Test de Z para dos proporciones 
**Nivel de significancia alpha:** 0.05


In [41]:
import statsmodels.api as sm
from statsmodels.stats.proportion import proportions_ztest

# Agrupamos los datos para obtener los éxitos (conversiones) y los totales por variante
resumen_experimento = df.groupby('variante')['convirtio'].agg(['sum', 'count'])

# Extraemos los valores para el test estadístico
conversiones = [resumen_experimento.loc['tratamiento', 'sum'], resumen_experimento.loc['control', 'sum']]
totales = [resumen_experimento.loc['tratamiento', 'count'], resumen_experimento.loc['control', 'count']]

# Calculamos las tasas de conversión descriptivas
tasa_control = (conversiones[1] / totales[1]) * 100
tasa_tratamiento = (conversiones[0] / totales[0]) * 100

print("--- RESULTADOS DESCRIPTIVOS ---")
print(f"Usuarios en Control: {totales[1]} | Conversiones: {conversiones[1]} | Tasa de Conversión: {tasa_control:.2f}%")
print(f"Usuarios en Tratamiento: {totales[0]} | Conversiones: {conversiones[0]} | Tasa de Conversión: {tasa_tratamiento:.2f}%")
print("\n" + "="*50 + "\n")

# Aplicamos el Z-test para dos proporciones
z_stat, p_value = proportions_ztest(count=conversiones, nobs=totales)

print("--- PRUEBA ESTADÍSTICA (Z-TEST) ---")
print(f"Estadístico Z (Z-score): {z_stat:.4f}")
print(f"Valor p (p-value): {p_value:.4f}")

# Conclusión formal según el nivel de significancia estándar (alpha = 0.05)
alpha = 0.05
print("\nInterpretación:")
if p_value < alpha:
    print("Rechazamos la hipótesis nula: hay evidencia de una diferencia.")
else:
    print("No rechazamos la hipótesis nula: no hay evidencia suficiente de una diferencia.")


--- RESULTADOS DESCRIPTIVOS ---
Usuarios en Control: 4965 | Conversiones: 779 | Tasa de Conversión: 15.69%
Usuarios en Tratamiento: 5035 | Conversiones: 820 | Tasa de Conversión: 16.29%


--- PRUEBA ESTADÍSTICA (Z-TEST) ---
Estadístico Z (Z-score): 0.8133
Valor p (p-value): 0.4161

Interpretación:
No rechazamos la hipótesis nula: no hay evidencia suficiente de una diferencia.


## 🧪 Paso 5: Evaluación de Impacto - Experimentación A/B

Para validar si las modificaciones en la interfaz de usuario (UI) del checkout realmente impactan la tasa de conversión de compra, realizamos una auditoría de calidad al dataset y ejecutamos un test estadístico de proporciones (Z-test).

### 🔍 1. Control de Calidad y Auditoría del Dataset (EDA)
Antes de proceder con la prueba estadística, se verificó la integridad de los datos para garantizar un resultado confiable:
* **Valores Faltantes/Nulos:** 0\% de nulos puros o valores centinela ocultos (`?`, `-999`).
* **Cardinalidad:** Consistencia absoluta en variantes (*control* y *tratamiento*), dispositivos y países.
* **Outliers:** La duración de las sesiones se comporta de manera normal, con un mínimo de 20 segundos y un máximo de 300 segundos, descartando anomalías por bots.
* **Dimensión Temporal:** El experimento cuenta con una distribución simétrica de 6 meses completos (del 1 de enero al 30 de junio de 2025).

---

### 📊 2. Planteamiento de Hipótesis y Resultados Descriptivos

* **Hipótesis Nula ($H_0$):** Los cambios en la UI del checkout **no tienen impacto**. La tasa de conversión del grupo tratamiento es igual o menor a la del grupo control.
* **Hipótesis Alternativa ($H_1$):** Los cambios en la UI del checkout **sí tienen un impacto positivo**. La tasa de conversión del grupo tratamiento es significativamente mayor que la del grupo control.

#### Muestras Evaluadas:
* **Grupo Control (UI Antigua):** 4,965 usuarios | 779 conversiones | **Tasa de Conversión: 15.69\%**
* **Grupo Tratamiento (UI Nueva):** 5,035 usuarios | 820 conversiones | **Tasa de Conversión: 16.29\%**

* **Diferencia Absoluta Observada:** +0.60\% a favor de la nueva interfaz.

---

### 📉 3. Resultado de la Prueba Estadística (Z-test)

* **Estadístico Z (Z-score):** 0.8133
* **Valor p (p-value Recalculado Unilateral):** 0.2080  
* **Nivel de Significancia ($\alpha$):** 0.05

**Interpretación Estadística:** Dado que nuestro **p-value (0.4161)** es ampliamente mayor que el nivel de significancia estándar ($\alpha = 0.05$), **no tenemos evidencia estadística suficiente para rechazar la hipótesis nula ($H_0$)**. 

Esto significa que el ligero incremento del 0.60\% en la conversión del grupo tratamiento se deba al azar o al ruido de los datos, y no al cambio de diseño en la interfaz.

---

### 🎯 4. Recomendaciones Finales para el Negocio

1. **Frenar la implementación global:** No se justifica el gasto de ingeniería, tiempo de desarrollo y soporte técnico para pasar la nueva UI a producción, ya que no asegura un retorno de inversión real.
2. **Análisis de segmentación profunda:** Se sugiere abrir el análisis por dimensiones. Es posible que la nueva UI funcione de manera espectacular en dispositivos *mobile* pero esté afectando la experiencia en *desktop*, o que su rendimiento varíe según el *país*.
3. **Auditar la Duración de la Sesión:** Se recomienda aplicar un T-test sobre la columna `duracion_sesion` para verificar si la nueva UI, aunque no aumente la conversión inmediata, logra que los usuarios finalicen su pago de forma más rápida (reducción de fricción temporal).

---

## 🔹 Paso 6: Comunicar los resultados (Dashboard en BI)

🎯 **Objetivo**:  
Crear un dashboard que muestre de manera clara y visual los resultados del análisis de ventas, costos, marketing y conversión. 

Se usarán los CSVs limpios del Paso 1:

- `orders_clean.csv`  
- `catalog_clean.csv`  
- `marketing_clean.csv`

---

1️⃣ Preparación de los datos
1. Cargar los CSVs en Power BI o Tableau.
2. Revisar relaciones:
   - `orders.nombre_producto` → `catalog.nombre_producto`
   - `orders.fecha_pedido` → tabla de fechas (crear calendario para análisis temporal)
   - `orders.fecha_pedido` → `dim_fecha.date`
3. Crear columnas calculadas necesarias
4. Crear tabla de fechas para poder calcular comparaciones YTD, YoY o períodos anteriores (`Previous Year`, `Previous Month`).

---

2️⃣ Dashboard 1: Overview Ejecutivo
**KPIs principales a mostrar:**
- Revenue total
- Profit total
- Gasto total en marketing
- Ticket promedio
- Cantidad promedio de productos por orden

**Visualizaciones sugeridas:**
- Tarjetas KPI para revenue, profit y gasto marketing
- Gráfico de líneas: evolución mensual de revenue o profit
- Gráfico de líneas YTD
- Gráfico de barras: revenue y profit por producto o categoría

---

 3️⃣ Dashboard 2: Detalle / Drill-through  
**Objetivo:** Permitir explorar los datos desde el KPI general hasta cada orden o producto.

**Visualizaciones sugeridas:**
- Tabla detallada de órdenes con:
  - producto, cantidad, revenue, cost, profit
  - color condicional (profit negativo en rojo, positivo en verde)
- Gráfico de barras por producto con medida `cantidad vendida`
- Drill-through: seleccionar un producto y ver todos los pedidos relacionados
- Filtros por fecha, categoría de producto, etc

---

## 🚀 Entrega Final

Comparte el acceso a tu Dashboard para revisión.   
Puedes entregar el Dashboard utilizando **Power BI o Tableau**.

Incluye **uno de los siguientes**:

- 🔗 Link público del dashboard publicado en **Power BI Service o Tableau Public / Tableau Cloud**
- 🔗 Link de **Google Drive o OneDrive** con el archivo del proyecto (`.pbix`) y los 3 csvs limpios.


### 📎 Enlace del Dashboard


https://drive.google.com/drive/folders/1nxJPCCPWVca8ThaH1zi-ZF-h0XPf7Aq9?usp=drive_link
